### Import libraries

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

import seaborn as sns
import matplotlib.pyplot as plt

import gc
import os

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

### Custom functions

In [ ]:
def pickle_dump(path, saveobj):
    import pickle
    filehandler = open(path,"wb")
    pickle.dump(saveobj,filehandler)
    print("File pickled")
    filehandler.close()

In [ ]:
def pickle_load(path):
    import pickle
    file = open(path,'rb')
    loadobj = pickle.load(file)
    file.close()
    return loadobj

In [ ]:
sampleSubmit_df = pd.read_csv("/kaggle/input/ncaaw-march-mania-2021/WSampleSubmissionStage1.csv")
season_df = pd.read_csv("/kaggle/input/ncaaw-march-mania-2021/WRegularSeasonCompactResults.csv")
tourney_df = pd.read_csv("/kaggle/input/ncaaw-march-mania-2021/WNCAATourneyCompactResults.csv")
seeds_df = pd.read_csv('../input/ncaaw-march-mania-2021/WNCAATourneySeeds.csv')
# ordinals_df = pd.read_csv('../input/ncaam-march-mania-2021/MMasseyOrdinals.csv')
teams_df = pd.read_csv('../input/ncaaw-march-mania-2021/WTeams.csv')

### Season Results Compact

In [ ]:
print(season_df.shape)
print("----------------------")
print(season_df.head())
print("----------------------")
print(season_df['WLoc'].value_counts())
print("----------------------")
print("Max DayNum: {}".format(season_df['DayNum'].max()))

### Prepare dataset for feature engineering 

In [ ]:
team1_df = tourney_df.copy()
team2_df = tourney_df.copy()

team1_df['Team'] = team1_df['WTeamID']
team1_df = team1_df.reset_index(drop=True)

team2_df['Team'] = team2_df['LTeamID']
team2_df = team2_df.reset_index(drop=True)

tourneyData_df = pd.concat([team1_df, team2_df], ignore_index=True)
tourneyData_df.drop_duplicates(inplace=True)
tourneyData_df = tourneyData_df.reset_index(drop=True)
tourneyData_df['gameType'] = 'NCAA'
tourneyData_df.shape

In [ ]:
team1_df = season_df.copy()
team2_df = season_df.copy()

team1_df['Team'] = team1_df['WTeamID']
team1_df = team1_df.reset_index(drop=True)

team2_df['Team'] = team2_df['LTeamID']
team2_df = team2_df.reset_index(drop=True)

seasonData_df = pd.concat([team1_df, team2_df], ignore_index=True)
seasonData_df.drop_duplicates(inplace=True)
seasonData_df = seasonData_df.reset_index(drop=True)
seasonData_df['gameType'] = 'Season'
seasonData_df.shape

In [ ]:
features_df = pd.concat([tourneyData_df, seasonData_df], ignore_index=True)
features_df.sort_values(by=['Team','Season','DayNum'], inplace=True)
features_df['Opp'] = np.where(features_df['Team'] != features_df['WTeamID'], features_df['WTeamID'], features_df['LTeamID'])
features_df = features_df.drop_duplicates().reset_index(drop=True)
features_df.shape

### Feature Engineering

### Wins, games and win ratios

In [ ]:
%%time

nwins = 0
nhomewins = 0
nawaywins = 0
neutralwins = 0
ngames = 0
nhomegames = 0
nawaygames = 0
neutralgames = 0

nwins_season = 0
nhomewins_season = 0
nawaywins_season = 0
neutralwins_season = 0
ngames_season = 0
nhomegames_season = 0
nawaygames_season = 0
neutralgames_season = 0

winRatio = 0
hWinRatio = 0
aWinRatio = 0
nWinRatio = 0

winRatio_season = 0
hWinRatio_season = 0
aWinRatio_season = 0
nWinRatio_season = 0

numWins = []
numHomeWins = []
numAwayWins = []
numNeutralWins = []
numGames = []
numHomeGames = []
numAwayGames = []
numNeutralGames = []

numWins_season = []
numHomeWins_season = []
numAwayWins_season = []
numNeutralWins_season = []
numGames_season = []
numHomeGames_season = []
numAwayGames_season = []
numNeutralGames_season = []

allWinRatio = []
homeWinRatio = []
awayWinRatio = []
neutralWinRatio = []

allWinRatio_season = []
homeWinRatio_season = []
awayWinRatio_season = []
neutralWinRatio_season = []

curTeam = 0
curSeason = 0

for row in features_df.itertuples():
    
    # Next Team
    if getattr(row,'Team') != curTeam:
        curTeam = getattr(row,'Team')
        curSeason = getattr(row,'Season')
        
        nwins = 0
        nhomewins = 0
        nawaywins = 0
        neutralwins = 0
        ngames = 0
        nhomegames = 0
        nawaygames = 0
        neutralgames = 0

        nwins_season = 0
        nhomewins_season = 0
        nawaywins_season = 0
        neutralwins_season = 0
        ngames_season = 0
        nhomegames_season = 0
        nawaygames_season = 0
        neutralgames_season = 0

        winRatio = 0
        hWinRatio = 0
        aWinRatio = 0
        nWinRatio = 0

        winRatio_season = 0
        hWinRatio_season = 0
        aWinRatio_season = 0
        nWinRatio_season = 0

    # Next Season
    elif getattr(row,'Season') != curSeason:
        curSeason = getattr(row,'Season')
        
        nwins_season = 0
        nhomewins_season = 0
        nawaywins_season = 0
        neutralwins_season = 0
        ngames_season = 0
        nhomegames_season = 0
        nawaygames_season = 0
        neutralgames_season = 0
        
        winRatio_season = 0
        hWinRatio_season = 0
        aWinRatio_season = 0
        nWinRatio_season = 0
    
    # Calculate feature value for row
    ngames+=1; ngames_season+=1;
    if getattr(row,'WLoc') == 'H': nhomegames+=1; nhomegames_season+=1;
    if getattr(row,'WLoc') == 'A': nawaygames+=1; nawaygames_season+=1;
    if getattr(row,'WLoc') == 'N': neutralgames+=1; neutralgames_season+=1;
    
    if getattr(row,'Team') == getattr(row,'WTeamID'): 
        nwins+=1; nwins_season+=1; 
        if getattr(row,'WLoc') == 'H': nhomewins+=1; nhomewins_season+=1;
        if getattr(row,'WLoc') == 'A': nawaywins+=1; nawaywins_season+=1;
        if getattr(row,'WLoc') == 'N': neutralwins+=1; neutralwins_season+=1;
    
    winRatio = 0 if ngames == 0 else nwins/ngames
    hWinRatio = 0 if nhomegames == 0 else nhomewins/nhomegames
    aWinRatio = 0 if nawaygames == 0 else nawaywins/nawaygames
    nWinRatio = 0 if neutralgames == 0 else neutralwins/neutralgames
    
    winRatio_season = 0 if ngames_season == 0 else nwins_season/ngames_season
    hWinRatio_season = 0 if nhomegames_season == 0 else nhomewins_season/nhomegames_season
    aWinRatio_season = 0 if nawaygames_season == 0 else nawaywins_season/nawaygames_season
    nWinRatio_season = 0 if neutralgames_season == 0 else neutralwins_season/neutralgames_season

    # Append feature values for row
    numWins.append(nwins)
    numHomeWins.append(nhomewins)
    numAwayWins.append(nawaywins)
    numNeutralWins.append(neutralwins)
    numGames.append(ngames)
    numHomeGames.append(nhomegames)
    numAwayGames.append(nawaygames)
    numNeutralGames.append(neutralgames)

    numWins_season.append(nwins_season)
    numHomeWins_season.append(nhomewins_season)
    numAwayWins_season.append(nawaywins_season)
    numNeutralWins_season.append(neutralwins_season)
    numGames_season.append(ngames_season)
    numHomeGames_season.append(nhomegames_season)
    numAwayGames_season.append(nawaygames_season)
    numNeutralGames_season.append(neutralgames_season)

    allWinRatio.append(winRatio)
    homeWinRatio.append(hWinRatio)
    awayWinRatio.append(aWinRatio)
    neutralWinRatio.append(nWinRatio)

    allWinRatio_season.append(winRatio_season)
    homeWinRatio_season.append(hWinRatio_season)
    awayWinRatio_season.append(aWinRatio_season)
    neutralWinRatio_season.append(nWinRatio_season)

In [ ]:
# Custom features
features_df['spy_numWins'] = numWins
features_df['spy_numHomeWins'] = numHomeWins
features_df['spy_numAwayWins'] = numAwayWins
features_df['spy_numNeutralWins'] = numNeutralWins
features_df['spy_numGames'] = numGames
features_df['spy_numHomeGames'] = numHomeGames
features_df['spy_numAwayGames'] = numAwayGames
features_df['spy_numNeutralGames'] = numNeutralGames

features_df['spy_numWins_season'] = numWins_season
features_df['spy_numHomeWins_season'] = numHomeWins_season
features_df['spy_numAwayWins_season'] = numAwayWins_season
features_df['spy_numNeutralWins_season'] = numNeutralWins_season
features_df['spy_numGames_season'] = numGames_season
features_df['spy_numHomeGames_season'] = numHomeGames_season
features_df['spy_numAwayGames_season'] = numAwayGames_season
features_df['spy_numNeutralGames_season'] = numNeutralGames_season

features_df['spy_allWinRatio'] = allWinRatio
features_df['spy_homeWinRatio'] = homeWinRatio
features_df['spy_awayWinRatio'] = awayWinRatio
features_df['spy_neutralWinRatio'] = neutralWinRatio

features_df['spy_allWinRatio_season'] = allWinRatio_season
features_df['spy_homeWinRatio_season'] = homeWinRatio_season
features_df['spy_awayWinRatio_season'] = awayWinRatio_season
features_df['spy_neutralWinRatio_season'] = neutralWinRatio_season

In [ ]:
features_df.head()

### Team head to head record

In [ ]:
features_df['isGame'] = 1

features_df['isWin'] = np.where(features_df['Team'] == features_df['WTeamID'], 1, 0)

features_df['isHome'] = np.where((features_df['Team'] == features_df['WTeamID']) & (features_df['WLoc']=='H'), 1,
                                np.where((features_df['Team'] == features_df['LTeamID']) & (features_df['WLoc']=='A'), 1,
                                        np.where((features_df['WLoc']=='N'), 0, 0)))

features_df['isHomeWin'] = np.where((features_df['isWin']==1) & (features_df['isHome']==1), 1, 0)

features_df['TeamH2H'] = np.where(features_df['Team'] == features_df['WTeamID'], 
                                  features_df['Team'].astype(str) + '-' + features_df['LTeamID'].astype(str),
                                  features_df['Team'].astype(str) + '-' + features_df['WTeamID'].astype(str))

features_df.head()

In [ ]:
temp = features_df.groupby('TeamH2H').agg(
            spy_numGames_H2H = ('isGame',sum),
            spy_numWins_H2H = ('isWin',sum),
            spy_homeGames_H2H = ('isHome',sum),
            spy_homeWins_H2H = ('isHomeWin',sum),
        ).reset_index(drop=False)

temp['spy_winRatio_H2H'] = np.where(temp['spy_numGames_H2H']!=0, temp['spy_numWins_H2H'] / temp['spy_numGames_H2H'], 0)
temp['spy_homeWinRatio_H2H'] = np.where(temp['spy_homeGames_H2H']!=0, temp['spy_homeWins_H2H'] / temp['spy_homeGames_H2H'], 0)

temp.head()

In [ ]:
features_df = pd.merge(features_df, temp, on="TeamH2H", how="left")
features_df.shape

In [ ]:
features_df.head()

### Seed ranking

In [ ]:
seeds_df['spy_SeedNum'] = (seeds_df['Seed'].str[1:3]).astype(int)
seed_feats_df = seeds_df.drop(columns=['Seed'])
seed_feats_df['spy_seeded'] = 1
seed_feats_df.rename(columns={'TeamID':'Team'}, inplace=True)
seed_feats_df['Season'] = seed_feats_df['Season']+1
seed_feats_df.head()

In [ ]:
features_df = pd.merge(features_df, seed_feats_df, on=['Season', 'Team'], how="left")

features_df['spy_SeedNum'] = features_df['spy_SeedNum'].fillna(20)
features_df['spy_seeded'] = features_df['spy_seeded'].fillna(0)
features_df.shape

### Create modeling dataset

In [ ]:
model_df1 = tourney_df[tourney_df['Season'] >= 2005].copy()
model_df2 = tourney_df[tourney_df['Season'] >= 2005].copy()

model_df1 = model_df1.loc[:, ['Season','WTeamID','LTeamID']]
model_df1.columns = ['Season','TeamID1','TeamID2']
model_df1['result'] = 1

model_df2 = model_df2.loc[:, ['Season','LTeamID','WTeamID']]
model_df2.columns = ['Season','TeamID1','TeamID2']
model_df2['result'] = 0

tourney_model_df = pd.concat([model_df1, model_df2])

del model_df1, model_df2
gc.collect()

tourney_model_df.head(1)

In [ ]:
model_df1 = season_df[season_df['Season'] >= 2005].copy()
model_df2 = season_df[season_df['Season'] >= 2005].copy()

model_df1 = model_df1.loc[:, ['Season','WTeamID','LTeamID']]
model_df1.columns = ['Season','TeamID1','TeamID2']
model_df1['result'] = 1

model_df2 = model_df2.loc[:, ['Season','LTeamID','WTeamID']]
model_df2.columns = ['Season','TeamID1','TeamID2']
model_df2['result'] = 0

season_model_df = pd.concat([model_df1, model_df2])

del model_df1, model_df2
gc.collect()

season_model_df.head(1)

In [ ]:
model_df = pd.concat([tourney_model_df, season_model_df])
model_df.shape

In [ ]:
print(model_df.shape)

train_df = pd.DataFrame()

for season in model_df.Season.unique():
    temp_df1 = features_df[(features_df['Season']==season) & (features_df['gameType']=='Season')].copy()
    temp_df2 = pd.DataFrame(temp_df1.groupby(['Team'])['DayNum'].max()).reset_index()
    featCols = [col for col in features_df.columns if 'spy_' in col]
    temp_df1 = temp_df1[['Team','DayNum','Season']+featCols]

    temp_df = pd.merge(temp_df2, temp_df1, on=['Team','DayNum'], how="left")
    
    team1_feats_df = temp_df.drop(['DayNum'], axis=1)
    team1_feats_df = team1_feats_df.add_prefix("Team1_")
    team1_feats_df.rename(columns = {'Team1_Team':'TeamID1','Team1_Season':'Season'}, inplace=True)

    team2_feats_df = temp_df.drop(['DayNum'], axis=1)
    team2_feats_df = team2_feats_df.add_prefix("Team2_")
    team2_feats_df.rename(columns = {'Team2_Team':'TeamID2','Team2_Season':'Season'}, inplace=True)
    
    model_df_temp = model_df[model_df['Season']==season].copy()

    model_df_temp = pd.merge(model_df_temp, team1_feats_df, on=['TeamID1','Season'], how="left")
    model_df_temp = pd.merge(model_df_temp, team2_feats_df, on=['TeamID2','Season'], how="left")
    
    train_df = train_df.append(model_df_temp)
    
print(train_df.shape)

In [ ]:
train_df.sort_values(by=['Season'], ascending=True, inplace=True)
train_df = train_df.reset_index(drop=True)
train_df.head()

In [ ]:
# Stat comparison features
train_df['spy_seedNumDiff'] = train_df['Team1_spy_SeedNum'] - train_df['Team2_spy_SeedNum']
train_df['spy_seededDiff'] = train_df['Team1_spy_seeded'] - train_df['Team2_spy_seeded']

# train_df['spy_maxOrdRnkDiff'] = train_df['Team1_spy_maxOrdinalRank'] - train_df['Team1_spy_maxOrdinalRank']
# train_df['spy_minOrdRnkDiff'] = train_df['Team1_spy_minOrdinalRank'] - train_df['Team1_spy_minOrdinalRank']
# train_df['spy_rangeOrdRnkDiff'] = train_df['Team1_spy_rangeOrdinalRank'] - train_df['Team1_spy_rangeOrdinalRank']
# train_df['spy_avgOrdRnkDiff'] = train_df['Team1_spy_avgOrdinalRank'] - train_df['Team1_spy_avgOrdinalRank']

train_df['spy_gameExpRatio'] = train_df['Team1_spy_numGames'] / train_df['Team2_spy_numGames']
train_df['spy_gameExpRatio_season'] = train_df['Team1_spy_numGames_season'] / train_df['Team2_spy_numGames_season']

train_df['spy_WinRecordRatio'] = train_df['Team1_spy_numWins'] / train_df['Team2_spy_numWins']
train_df['spy_WinRecordRatio_season'] = train_df['Team1_spy_numWins_season'] / train_df['Team2_spy_numWins_season']

train_df['spy_WinRatioRatio'] = train_df['Team1_spy_allWinRatio'] / train_df['Team2_spy_allWinRatio']
train_df['spy_WinRatioRatio_season'] = train_df['Team1_spy_allWinRatio_season'] / train_df['Team2_spy_allWinRatio_season']

In [ ]:
train_df['spy_numWinsRatio_H2H'] = np.where(train_df['Team2_spy_numWins_H2H'] != 0, 
                                            train_df['Team1_spy_numWins_H2H'] / train_df['Team2_spy_numWins_H2H'], 0)
train_df['spy_numHomeGamesRatio_H2H'] = np.where(train_df['Team2_spy_homeGames_H2H'] != 0, 
                                            train_df['Team1_spy_homeGames_H2H'] / train_df['Team2_spy_homeGames_H2H'], 0)
train_df['spy_numHomeWinsRatio_H2H'] = np.where(train_df['Team2_spy_homeWins_H2H'] != 0, 
                                            train_df['Team1_spy_homeWins_H2H'] / train_df['Team2_spy_homeWins_H2H'], 0)
train_df['spy_WinRatioRatio_H2H'] = np.where(train_df['Team2_spy_winRatio_H2H'] != 0, 
                                            train_df['Team1_spy_winRatio_H2H'] / train_df['Team2_spy_winRatio_H2H'], 0)
train_df['spy_homeWinRatioRatio_H2H'] = np.where(train_df['Team2_spy_homeWinRatio_H2H'] != 0, 
                                            train_df['Team1_spy_homeWinRatio_H2H'] / train_df['Team2_spy_homeWinRatio_H2H'], 0)

In [ ]:
print(train_df.shape)
train_df = train_df.dropna()
print(train_df.shape)

In [ ]:
train_df1 = train_df[(train_df['Team1_spy_SeedNum']!=20) & (train_df['Team2_spy_SeedNum']!=20)].copy()
print(train_df1.shape)
train_df2 = train_df[(train_df['Team1_spy_SeedNum']==20) | (train_df['Team2_spy_SeedNum']==20)].copy()
train_df2 = train_df2.sample(5000)
print(train_df2.shape)

train_df = pd.concat([train_df1, train_df2]).drop_duplicates().reset_index(drop=True)
print(train_df.shape)